# Honda Civic Type R RAG Assistant (Colab)

This notebook is the answer-generation half of the project. Retrieval is built in the repository (`src/`). This notebook adds a small open-source language model (Qwen2.5-3B-Instruct) that writes an answer from the retrieved manual excerpts, running on Colab's free T4 GPU. No API key and no billing are needed.

**Before running**

1. Runtime > Change runtime type > **T4 GPU**
2. Build `data/manual_chunks.json` locally from your own copy of the manual (`python src/extract.py` then `python src/chunk.py`). It is not in the repository because the manual is Honda's copyrighted material.
3. Run the cells in order and upload `manual_chunks.json` when asked.

In [ ]:
!pip install -q transformers accelerate chromadb

## 1. Upload the chunks

In [ ]:
from google.colab import files

uploaded = files.upload()
CHUNKS_FILE = next(iter(uploaded))
print(CHUNKS_FILE)

## 2. Build the vector database

In [ ]:
import json
import chromadb

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

client = chromadb.Client()
try:
    client.delete_collection("civic_manual")
except Exception:
    pass
collection = client.create_collection(name="civic_manual")

collection.add(
    ids=[str(c["chunk_id"]) for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"start_page": c["start_page"], "end_page": c["end_page"]} for c in chunks],
)
print(f"Stored {collection.count()} chunks")

## 3. Load the language model

In [ ]:
import torch
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto",
)
print("Model loaded")

## 4. Retrieve, then answer

In [ ]:
def ask(question, n_results=4):
    results = collection.query(query_texts=[question], n_results=n_results)
    metadatas = results["metadatas"][0]

    blocks = []
    for text, meta in zip(results["documents"][0], metadatas):
        blocks.append(f"[pages {meta['start_page']}-{meta['end_page']}]\n{text}")
    context = "\n\n---\n\n".join(blocks)

    prompt = f"""You are a helpful assistant answering questions about a Honda Civic Type R using only the owner's manual excerpts below.
Only use information that appears in the excerpts. If the excerpts do not contain the answer, say you could not find it in the retrieved sections instead of guessing.
Cite the page numbers your answer comes from.

MANUAL EXCERPTS:
{context}

QUESTION:
{question}"""

    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=300, do_sample=False)
    answer = output[0]["generated_text"][-1]["content"]

    pages = ", ".join(f"{m['start_page']}-{m['end_page']}" for m in metadatas)
    return answer + f"\n\nRetrieved page ranges: {pages}"

## 5. Example questions

In [ ]:
for q in [
    "What is the recommended tyre pressure?",
    "How do I reset the maintenance light?",
    "What does the airbag light mean?",
]:
    print("Q:", q)
    print(ask(q))
    print("-" * 60)

## Known limitation

The engine oil question still fails. The answer is in a multi-column specifications table (page 765) that text extraction scrambles, so the right chunk is not retrieved. The model then answers from the general oil guidance it did retrieve and leaves out the specific grade (0W-20), so the answer is grounded but incomplete. Run the cell below to see this.

In [ ]:
print(ask("What type of engine oil should I use?"))

## Ask your own

In [ ]:
question = input("Ask a question: ")
print(ask(question))